<a href="https://colab.research.google.com/github/KingTechnician/shared-truth/blob/main/tiu_calibrate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/KingTechnician/Truth_is_Universal.git

fatal: destination path 'Truth_is_Universal' already exists and is not an empty directory.


In [2]:
import os
import glob
import sys

USE_COLAB = False 

if USE_COLAB:

    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

else:
    from dotenv import load_dotenv
    load_dotenv()

# Optional: Verify the token is active
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

/home/kingtechnician/st-research/shared-truth-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Import Truth is Universal libraries

In [4]:
tiu_path = "/home/kingtechnician/st-research/Truth_is_Universal"

%cd $tiu_path


repo_path = '/content/Truth_is_Universal'
if repo_path not in sys.path:
    sys.path.append(repo_path)

try:
    import utils
    import calibration
    print("✅ Successfully imported Truth_is_Universal libraries.")

except ImportError as e:
    print(f"❌ Error importing libraries: {e}")
    print("Verify that the files (utils.py, generate_acts.py, probes.py) exist in " + repo_path)

/home/kingtechnician/st-research/Truth_is_Universal
✅ Successfully imported Truth_is_Universal libraries.


Run their generate_acts script to generate activations at ALL layers

In [5]:
!python generate_acts.py \
  --model_family Gemma \
  --model_size 7B \
  --model_type instruct \
  --layers -1 \
  --datasets cities neg_cities sp_en_trans neg_sp_en_trans \
  --output_dir acts \
  --device cuda:0

config.json: 100%|█████████████████████████████| 694/694 [00:00<00:00, 6.71MB/s]
tokenizer_config.json: 100%|███████████████| 34.2k/34.2k [00:00<00:00, 6.99MB/s]
tokenizer.json: 100%|██████████████████████| 17.5M/17.5M [00:01<00:00, 11.7MB/s]
special_tokens_map.json: 100%|█████████████████| 636/636 [00:00<00:00, 7.99MB/s]
model.safetensors.index.json: 100%|████████| 20.9k/20.9k [00:00<00:00, 4.26MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [03:43<00:00, 55.88s/it]
Download complete: 100%|███████████████████| 17.1G/17.1G [03:43<00:00, 76.4MB/s]
Loading weights: 100%|█| 254/254 [00:00<00:00, 3387.35it/s, Materializing param=
generation_config.json: 100%|██████████████████| 137/137 [00:00<00:00, 1.81MB/s]
/home/kingtechnician/st-research/shared-truth-env/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0)

Use the calibration library to compute for the best truth layer within the model. Run a few times to get the most ranked layer - sometimes it will fluctuate.

We use a sample of the dataset and this seems enough for performant truth classification. Test with your own linear probes to verify.

In [6]:
model = "Gemma"
params = "7B"
version = "instruct"

candidate_layers = calibration.infer_available_layers(model,params,version)

best_layer, layer_scores = calibration.calibrate_best_layer(
    model_family=model,
    model_size=params,
    model_type=version,
    layers=candidate_layers,
    datasets=["cities", "neg_cities", "sp_en_trans", "neg_sp_en_trans"],
    samples_per_dataset=50,
)
print("Best layer:", best_layer)


Found layers on disk for Gemma-7B-instruct: 0..27
Layer 0 label stats: unique = (tensor([0., 1.]), tensor([100, 100]))
Layer   0: separation score = 2.495477e-04


/home/kingtechnician/st-research/Truth_is_Universal/utils.py:70: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  labels = t.Tensor(df[label].values).to(device)


Layer 1 label stats: unique = (tensor([0., 1.]), tensor([100, 100]))
Layer   1: separation score = 1.823982e-04
Layer 2 label stats: unique = (tensor([0., 1.]), tensor([100, 100]))
Layer   2: separation score = 5.417605e-04
Layer 3 label stats: unique = (tensor([0., 1.]), tensor([100, 100]))
Layer   3: separation score = 6.815090e-04
Layer 4 label stats: unique = (tensor([0., 1.]), tensor([100, 100]))
Layer   4: separation score = 1.390495e-03
Layer 5 label stats: unique = (tensor([0., 1.]), tensor([100, 100]))
Layer   5: separation score = 2.413707e-03
Layer 6 label stats: unique = (tensor([0., 1.]), tensor([100, 100]))
Layer   6: separation score = 3.851226e-03
Layer 7 label stats: unique = (tensor([0., 1.]), tensor([100, 100]))
Layer   7: separation score = 8.396702e-03
Layer 8 label stats: unique = (tensor([0., 1.]), tensor([100, 100]))
Layer   8: separation score = 8.015200e-03
Layer 9 label stats: unique = (tensor([0., 1.]), tensor([100, 100]))
Layer   9: separation score = 9.676